# Notebook 02: Web scraping de reseñas en español (SensaCine)

Obtiene críticas de espectadores **escritas originalmente en español** y las guarda en MongoDB, junto a las del Proyecto 1.

**Fuente:** SensaCine (`sensacine.com`). Se intentó primero con FilmAffinity, pero el sitio respondió **HTTP 429** (demasiadas solicitudes) al código aunque abría normal en el navegador, así que se cambió de fuente y se documentó, como pide el enunciado.

Reglas que cumple el scraper (`scraper.py`):
- Lee y respeta `robots.txt` antes de cada solicitud.
- Pausa de 2 a 3,5 segundos entre solicitudes y `User-Agent` identificatorio.
- Si el sitio responde 429 la pausa se duplica, y si falla varias veces seguidas el scraper se detiene. Si bloquea (403), se detiene y se cambia de fuente. No se evade ningún bloqueo.
- No se guardan nombres de usuario: el extractor no lee el bloque del autor.
- Cada página se guarda apenas se obtiene, así que un corte no pierde el progreso.

## 1. Configuración

In [1]:
import os
import sys

sys.path.append("..")  # por si se usa la estructura del repo (con src/)

try:
    from src.db_manager import get_db, crear_indices, conteo_por, genero_por_origen
    from src.scraper import (ScraperEducado, BloqueadoError, recolectar, parsear_resenas_sensacine,
                             parsear_obra_sensacine, parsear_listado_sensacine, url_resenas_sensacine,
                             obras_por_genero_sensacine, ids_obras_guardadas)
    from src.migracion import construir_actualizaciones
    from src.preprocessing import cargar_spacy, entrenar_etiquetador_nltk, procesar_resenas
except ImportError:  # archivos en la misma carpeta que el notebook
    from db_manager import get_db, crear_indices, conteo_por, genero_por_origen
    from scraper import (ScraperEducado, BloqueadoError, recolectar, parsear_resenas_sensacine,
                         parsear_obra_sensacine, parsear_listado_sensacine, url_resenas_sensacine,
                         obras_por_genero_sensacine, ids_obras_guardadas)
    from migracion import construir_actualizaciones
    from preprocessing import cargar_spacy, entrenar_etiquetador_nltk, procesar_resenas

FUENTE = "sensacine"               # identifica el sitio en el campo "fuente"
BASE = "https://www.sensacine.com"
ESCALA_MAX = 5                     # SensaCine: estrellas de 0,5 a 5 (se multiplican por 2)

db = get_db()
crear_indices(db)
col = db["resenas"]
print("Reseñas actuales en MongoDB:", col.count_documents({}))

Reseñas actuales en MongoDB: 7163


## 2. Prueba de acceso

Antes de scrapear, se comprueba si el sitio deja entrar al código.

- Si imprime el HTML de la página: se puede continuar.
- Si dice **Bloqueado** o sale HTTP 429 de forma repetida: no insistir. Esperar o cambiar de fuente y documentarlo en el README.
- No se debe intentar disfrazar el script ni usar Selenium para esquivar un bloqueo.

In [2]:
URL_PRUEBA = BASE + "/peliculas/todas-peliculas/genero-13009/"

try:
    scraper = ScraperEducado(BASE)
    print("¿robots.txt lo permite?", scraper.permitido(URL_PRUEBA))
    html = scraper.get(URL_PRUEBA)
    print("Largo del HTML:", len(html or ""))
    print((html or "")[:300])
except BloqueadoError as e:
    print("Bloqueado:", e)

¿robots.txt lo permite? True
Largo del HTML: 467202
<!DOCTYPE html>
<html lang="es">
    <head>
                <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">

        <meta charset="UTF-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no">

     


## 3. Extracción de las reseñas

Los extractores (en `scraper.py`) se escribieron con páginas reales de SensaCine y **no leen el autor**, así que no se guardan nombres de usuario:

- `parsear_resenas_sensacine`: críticas de espectadores (10 por página). La nota va en estrellas de 0,5 a 5.
- `parsear_listado_sensacine`: películas de un género (15 por página), con título, géneros y año. Solo se toman las que ya tienen nota de usuarios.
- Las URLs se arman con el id de la película, porque SensaCine esconde los enlaces internos en sus páginas.

La celda siguiente prueba los extractores con los archivos `sc_resenas.html` y `sc_lista.html` guardados antes (no los subas a GitHub: traen nombres de usuario).

In [3]:
with open("sc_resenas.html", encoding="utf-8") as f:
    html_resenas = f.read()
with open("sc_lista.html", encoding="utf-8") as f:
    html_lista = f.read()

print(parsear_obra_sensacine(html_resenas))
for r in parsear_resenas_sensacine(html_resenas)[:3]:
    print(r["id_sitio"], r["calificacion"], r["fecha"], len(r["texto"].split()), "palabras")

titulo, peliculas = parsear_listado_sensacine(html_lista)
print(titulo, "|", len(peliculas), "películas")
print(peliculas[0])

{'titulo_obra': '2012', 'generos': [], 'anio_estreno': None, 'tipo_obra': 'pelicula'}
85535 1.0 2010-08-10 00:00:00 31 palabras
1005634198 3.0 2020-04-13 00:00:00 117 palabras
49787 1.0 2009-12-17 00:00:00 49 palabras
Películas de acción recomendadas - SensaCine.com | 15 películas
{'url': 'https://www.sensacine.com/peliculas/pelicula-1000013045/', 'titulo': 'La Odisea', 'generos': ['Acción', 'Fantasía'], 'anio_estreno': 2026, 'con_nota_usuarios': True}


## 4. Géneros y metas

El scraper arma solo la lista de películas: recorre el listado de cada género (por popularidad, 15 por página) y de cada película toma sus críticas de espectadores. **El género de las reseñas es el del listado del que salió la película**; la lista completa de géneros de la película queda en el campo `generos`.

- `GENEROS`: código del género según el menú de SensaCine y su nombre.
- `META_POR_GENERO`: reseñas nuevas que se buscan por género. El enunciado pide al menos 100 por género.
- `MAX_PAGINAS`: páginas de críticas por película (10 reseñas por página).
- `MAX_OBRAS`: películas que se consideran por género.

**Para probar primero:** `META_POR_GENERO = 50` y un solo género en `GENEROS`.

In [4]:
GENEROS = {"13008": "Drama", "13005": "Comedia", "13025": "Acción", "13009": "Terror", "13026": "Animación"}
META_POR_GENERO = 1000
MAX_PAGINAS = 20
MAX_OBRAS = 150

## 5. Recolección

Recorre los géneros uno por uno y guarda cada página en MongoDB apenas la obtiene. Se puede detener y volver a correr: las películas que ya tienen reseñas guardadas se omiten y las reseñas repetidas no se duplican. Una película interrumpida a la mitad no se retoma.

In [5]:
try:
    scraper = ScraperEducado(BASE)
    for codigo, nombre in GENEROS.items():
        print(f"=== {nombre} ===")
        obras = obras_por_genero_sensacine(scraper, codigo, nombre, max_obras=MAX_OBRAS)
        recolectar(db, scraper, FUENTE, obras, url_resenas_sensacine,
                   parsear_resenas_sensacine, parsear_obra_sensacine, ESCALA_MAX, MAX_PAGINAS,
                   max_nuevas=META_POR_GENERO, omitir=ids_obras_guardadas(db, FUENTE))
except BloqueadoError as e:
    print("El sitio bloqueó o rechazó el acceso:", e)
    print("Espera un buen rato antes de reintentar, o cambia de fuente y documéntalo.")
except PermissionError as e:
    print("robots.txt no permite una de las rutas:", e)

=== Drama ===
  Aviso: la página dice 'Películas dramáticas recomendados - SensaCine.com'; comprueba que el código 13008 sea Drama.
La Bola Negra: 58 reseñas nuevas en total
Ruega por nosotras: 60 reseñas nuevas en total
Clayface: 65 reseñas nuevas en total
Whalefall: 68 reseñas nuevas en total
Verity: La sombra de un engaño: 71 reseñas nuevas en total
3 metros sobre el cielo: 158 reseñas nuevas en total
El ser querido: 168 reseñas nuevas en total
Sacamantecas: 169 reseñas nuevas en total
Sentido y sensibilidad: 170 reseñas nuevas en total
Wicker: 170 reseñas nuevas en total
El precio de la red: 173 reseñas nuevas en total
Fiesta Pagäna: 173 reseñas nuevas en total
Cincuenta sombras de Grey: 360 reseñas nuevas en total
Lolita: 368 reseñas nuevas en total
Cronos: 374 reseñas nuevas en total
365 días: 388 reseñas nuevas en total
Culpa tuya: 406 reseñas nuevas en total
Drive – The Pretenders: 406 reseñas nuevas en total
Lolita: 408 reseñas nuevas en total
Los juegos del hambre: 517 reseña

## 6. Verificación

Metas del enunciado: reseñas nuevas por scraping (1.000 según el texto y la rúbrica; la tabla 2.3 menciona 3.000, hay que confirmar con el profesor), **5 géneros con al menos 100 reseñas de scraping cada uno** y todas con calificación.

In [6]:
print("Total en MongoDB:", col.count_documents({}))
print("Por fuente:", conteo_por(db, "fuente"))
print("Género x origen:", genero_por_origen(db))
print("Nativas sin calificación:", col.count_documents({"origen_texto": "nativo", "calificacion": {"$exists": False}}))
print("Nativas por género:", list(col.aggregate([
    {"$match": {"origen_texto": "nativo"}},
    {"$group": {"_id": "$genero_principal", "n": {"$sum": 1}}}, {"$sort": {"n": -1}}])))

Total en MongoDB: 12375
Por fuente: [{'_id': 'imdb_kaggle', 'n': 6283}, {'_id': 'sensacine', 'n': 5212}, {'_id': 'filmaffinity', 'n': 880}]
Género x origen: [{'_id': {'genero': 'Acción', 'origen': 'nativo'}, 'n': 1424}, {'_id': {'genero': 'Acción', 'origen': 'traducido'}, 'n': 1256}, {'_id': {'genero': 'Animación', 'origen': 'nativo'}, 'n': 1015}, {'_id': {'genero': 'Animación', 'origen': 'traducido'}, 'n': 1258}, {'_id': {'genero': 'Comedia', 'origen': 'nativo'}, 'n': 1006}, {'_id': {'genero': 'Comedia', 'origen': 'traducido'}, 'n': 1257}, {'_id': {'genero': 'Drama', 'origen': 'nativo'}, 'n': 1522}, {'_id': {'genero': 'Drama', 'origen': 'traducido'}, 'n': 1261}, {'_id': {'genero': 'Terror', 'origen': 'nativo'}, 'n': 1125}, {'_id': {'genero': 'Terror', 'origen': 'traducido'}, 'n': 1251}]
Nativas sin calificación: 0
Nativas por género: [{'_id': 'Drama', 'n': 1522}, {'_id': 'Acción', 'n': 1424}, {'_id': 'Terror', 'n': 1125}, {'_id': 'Animación', 'n': 1015}, {'_id': 'Comedia', 'n': 1006}]

## 7. Etiquetado de las reseñas nuevas

Las reseñas del scraping pasan por el mismo etiquetado del Proyecto 1 (spaCy y NLTK entrenado con cess_esp), para que todo el corpus tenga `pos_tags` y `metricas`. Requiere `spacy` (con `es_core_news_md`) y `nltk`.

In [9]:
from pymongo import UpdateOne

pendientes = list(col.find({"origen_texto": "nativo", "pos_tags": {"$exists": False}},
                           {"_id": 0, "id_resena": 1, "texto_es": 1}))
print("Reseñas por etiquetar:", len(pendientes))

if pendientes:
    nlp = cargar_spacy()
    bigrama, precision = entrenar_etiquetador_nltk()
    print("Precisión del etiquetador NLTK:", precision)

    textos = {d["id_resena"]: d["texto_es"] for d in pendientes}
    tokens = procesar_resenas(list(textos), list(textos.values()), nlp, bigrama)
    acts = construir_actualizaciones(textos, tokens)

    col.bulk_write([UpdateOne({"id_resena": k}, {"$set": v}) for k, v in acts.items()])
    print("Reseñas etiquetadas:", len(acts))

Reseñas por etiquetar: 6092
Precisión del etiquetador NLTK: {'unigrama': 0.8622152746761947, 'bigrama': 0.8761351793955635}
Reseñas etiquetadas: 6085
